# Advanced Generator States — A Tutorial with Problems and Complete Solutions

This notebook continues the original **Generator States** lesson in the same tutorial style: one idea, one small experiment, and one conclusion at a time.

Instead of repeating the original examples, we will investigate new and more advanced behavior:

- values flowing *into* a suspended generator;
- return values flowing *out of* a closed generator;
- exception injection with `throw()`;
- deterministic cleanup with `close()`;
- delegation with `yield from`;
- generator-frame introspection;
- re-entrancy errors;
- lazy pull-based execution;
- generator-backed context managers;
- finite-state machines and cooperative schedulers.

Every problem includes a worked solution and executable checks.

## Notebook conventions

A generator can be in one of four states:

- `GEN_CREATED`
- `GEN_RUNNING`
- `GEN_SUSPENDED`
- `GEN_CLOSED`

We will use `inspect.getgeneratorstate()` throughout the notebook.

In [1]:
from __future__ import annotations

from collections import deque
from collections.abc import Callable, Generator, Iterable, Iterator
from contextlib import contextmanager
from inspect import getgeneratorstate
from itertools import islice
from typing import Any, TypeVar

T = TypeVar("T")

## 1. Rebuilding the state timeline

Let's begin with a generator that records when its body actually starts running.

In [2]:
events: list[str] = []

def labeled_letters(text: str) -> Iterator[str]:
    events.append("body started")
    for letter in text:
        events.append(f"about to yield {letter}")
        yield letter
        events.append(f"resumed after {letter}")
    events.append("body returned")

Calling the generator function creates the generator object, but does not run the body.

In [3]:
g = labeled_letters("AB")
getgeneratorstate(g), events

('GEN_CREATED', [])

The first `next()` changes the state from created to running, executes until `yield`, and leaves the generator suspended.

In [4]:
first = next(g)
first, getgeneratorstate(g), events

('A', 'GEN_SUSPENDED', ['body started', 'about to yield A'])

The second `next()` resumes immediately after the previous `yield`.

In [5]:
second = next(g)
second, getgeneratorstate(g), events

('B',
 'GEN_SUSPENDED',
 ['body started', 'about to yield A', 'resumed after A', 'about to yield B'])

One more resume lets the function return. At that point the state becomes closed.

In [6]:
try:
    next(g)
except StopIteration:
    pass

getgeneratorstate(g), events

('GEN_CLOSED',
 ['body started',
  'about to yield A',
  'resumed after A',
  'about to yield B',
  'resumed after B',
  'body returned'])

### Problem 1 — Produce a complete state history

Write `state_history(generator)`.

It should return a list of dictionaries. Each dictionary should contain:

- the state immediately before a pull;
- the yielded value, when one exists;
- the state immediately after the pull.

The final record should show the transition to `GEN_CLOSED`.

### Solution

In [7]:
def state_history(generator: Iterator[T]) -> list[dict[str, object]]:
    history: list[dict[str, object]] = []

    while True:
        before = getgeneratorstate(generator)
        try:
            value = next(generator)
        except StopIteration:
            history.append(
                {
                    "before": before,
                    "value": None,
                    "after": getgeneratorstate(generator),
                }
            )
            return history
        else:
            history.append(
                {
                    "before": before,
                    "value": value,
                    "after": getgeneratorstate(generator),
                }
            )


history = state_history(labeled_letters("XY"))
history

[{'before': 'GEN_CREATED', 'value': 'X', 'after': 'GEN_SUSPENDED'},
 {'before': 'GEN_SUSPENDED', 'value': 'Y', 'after': 'GEN_SUSPENDED'},
 {'before': 'GEN_SUSPENDED', 'value': None, 'after': 'GEN_CLOSED'}]

In [8]:
assert history == [
    {"before": "GEN_CREATED", "value": "X", "after": "GEN_SUSPENDED"},
    {"before": "GEN_SUSPENDED", "value": "Y", "after": "GEN_SUSPENDED"},
    {"before": "GEN_SUSPENDED", "value": None, "after": "GEN_CLOSED"},
]

## 2. A `yield` expression can receive a value

So far, we have treated `yield` as an output statement.

But `yield` is also an expression. When the generator resumes through `send(value)`, that value becomes the result of the suspended `yield` expression.

In [9]:
def one_message_conversation() -> Generator[str, str, str]:
    reply = yield "ready"
    yield f"heard: {reply}"
    return "conversation finished"

The generator must first reach its initial `yield`. This is often called **priming** the generator.

In [10]:
conversation = one_message_conversation()
getgeneratorstate(conversation)

'GEN_CREATED'

In [11]:
next(conversation)

'ready'

Now the generator is suspended at `reply = yield "ready"`. Sending a value assigns that value to `reply`.

In [12]:
conversation.send("hello")

'heard: hello'

The next resume reaches the `return` statement. The returned value is attached to `StopIteration`.

In [13]:
try:
    next(conversation)
except StopIteration as exc:
    conversation_result = exc.value

conversation_result, getgeneratorstate(conversation)

('conversation finished', 'GEN_CLOSED')

### Problem 2 — Build a multiplier protocol

Create `multiplier_protocol(factor)`.

Protocol:

1. Its first yielded value is the string `"READY"`.
2. Each number sent to it is multiplied by `factor` and yielded.
3. Sending the string `"STOP"` closes the protocol by returning the number of processed inputs.

### Solution

In [14]:
def multiplier_protocol(
    factor: float,
) -> Generator[str | float, float | str, int]:
    processed = 0
    incoming = yield "READY"

    while incoming != "STOP":
        processed += 1
        incoming = yield float(incoming) * factor

    return processed

In [15]:
multiplier = multiplier_protocol(2.5)
assert next(multiplier) == "READY"
assert multiplier.send(4) == 10.0
assert multiplier.send(-2) == -5.0

try:
    multiplier.send("STOP")
except StopIteration as exc:
    processed_inputs = exc.value

processed_inputs, getgeneratorstate(multiplier)

(2, 'GEN_CLOSED')

In [16]:
assert processed_inputs == 2
assert getgeneratorstate(multiplier) == "GEN_CLOSED"

## 3. Why a newly created generator accepts only `None`

A newly created generator has not yet reached a `yield` expression.

Therefore, there is nowhere for a non-`None` value to go. Sending a non-`None` value before priming raises `TypeError`.

In [17]:
unprimed = one_message_conversation()

try:
    unprimed.send("too early")
except TypeError as exc:
    first_send_error = str(exc)

first_send_error, getgeneratorstate(unprimed)

("can't send non-None value to a just-started generator", 'GEN_CREATED')

Notice that this error does not start or close the generator. It remains created.

### Problem 3 — Write a safe priming helper

Create `prime(generator)`.

Requirements:

- the argument must be in `GEN_CREATED` state;
- it should call `next()` exactly once;
- it should return both the generator and its first yielded value;
- it should reject already-started generators.

### Solution

In [18]:
def prime(generator: Generator[T, Any, Any]) -> tuple[Generator[T, Any, Any], T]:
    state = getgeneratorstate(generator)
    if state != "GEN_CREATED":
        raise ValueError(f"expected GEN_CREATED, received {state}")

    first_value = next(generator)
    return generator, first_value

In [19]:
primed_conversation, greeting = prime(one_message_conversation())
assert greeting == "ready"
assert getgeneratorstate(primed_conversation) == "GEN_SUSPENDED"

try:
    prime(primed_conversation)
except ValueError as exc:
    repeated_prime_error = str(exc)

repeated_prime_error

'expected GEN_CREATED, received GEN_SUSPENDED'

## 4. Returning a value from a generator

A normal function returns directly to its caller.

A generator returns when it closes. Its return value becomes `StopIteration.value`.

In [20]:
def words_with_count(sentence: str) -> Generator[str, None, int]:
    words = sentence.split()
    for word in words:
        yield word.lower()
    return len(words)

In [21]:
word_generator = words_with_count("Generators keep execution state")
next(word_generator)

'generators'

A `for` loop hides the final `StopIteration`, so it also hides the return value.

In [22]:
list(word_generator)

['keep', 'execution', 'state']

To preserve the return value, consume the generator manually.

### Problem 4 — Consume values and preserve the final result

Write `consume_with_result(generator)` that returns `(yielded_values, return_value)`.

### Solution

In [23]:
def consume_with_result(
    generator: Generator[T, Any, Any],
) -> tuple[list[T], Any]:
    values: list[T] = []

    while True:
        try:
            values.append(next(generator))
        except StopIteration as exc:
            return values, exc.value

In [24]:
normalized_words, word_count = consume_with_result(
    words_with_count("One TWO three FOUR")
)

normalized_words, word_count

(['one', 'two', 'three', 'four'], 4)

In [25]:
assert normalized_words == ["one", "two", "three", "four"]
assert word_count == 4

## 5. `throw()` injects an exception at the suspension point

Calling `generator.throw(SomeException)` resumes the generator by raising that exception exactly where it is suspended.

The generator may catch the exception and continue, or let it escape and close.

In [26]:
class ResetTotal(Exception):
    """Reset an accumulator without closing it."""


def resettable_total() -> Generator[int, int, None]:
    total = 0

    while True:
        try:
            incoming = yield total
        except ResetTotal:
            total = 0
        else:
            total += incoming

In [27]:
totaler = resettable_total()
next(totaler)

0

In [28]:
totaler.send(10), totaler.send(7)

(10, 17)

The exception is handled inside the generator, so the generator remains suspended and usable.

In [29]:
totaler.throw(ResetTotal), getgeneratorstate(totaler)

(0, 'GEN_SUSPENDED')

In [30]:
totaler.send(3)

3

### Problem 5 — Add two recoverable commands

Create `controlled_stack()`.

It should yield a snapshot tuple of the current stack.

- Sending a normal value pushes it.
- Throwing `PopValue` removes the most recent item, if one exists.
- Throwing `ClearStack` removes every item.

### Solution

In [31]:
class PopValue(Exception):
    pass


class ClearStack(Exception):
    pass


def controlled_stack() -> Generator[tuple[object, ...], object, None]:
    stack: list[object] = []

    while True:
        try:
            incoming = yield tuple(stack)
        except PopValue:
            if stack:
                stack.pop()
        except ClearStack:
            stack.clear()
        else:
            stack.append(incoming)

In [32]:
stack = controlled_stack()
assert next(stack) == ()
assert stack.send("A") == ("A",)
assert stack.send("B") == ("A", "B")
assert stack.throw(PopValue) == ("A",)
assert stack.send("C") == ("A", "C")
assert stack.throw(ClearStack) == ()
assert getgeneratorstate(stack) == "GEN_SUSPENDED"
stack.close()

## 6. Unhandled injected exceptions close the generator

If a thrown exception is not caught inside the generator, it propagates to the caller and closes the generator.

In [ ]:
def simple_counter() -> Iterator[int]:
    number = 0
    while True:
        yield number
        number += 1

counter = simple_counter()
assert next(counter) == 0

try:
    counter.throw(RuntimeError, "injected failure")
except RuntimeError as exc:
    injected_error = str(exc)

injected_error, getgeneratorstate(counter)

### Problem 6 — Distinguish recoverable and fatal exceptions

Build `guarded_counter()`.

- `SkipIncrement` should be handled and keep the generator alive.
- Any other exception should close it naturally.

### Solution

In [34]:
class SkipIncrement(Exception):
    pass


def guarded_counter() -> Iterator[int]:
    number = 0
    while True:
        try:
            yield number
        except SkipIncrement:
            continue
        number += 1

In [35]:
guarded = guarded_counter()
assert next(guarded) == 0
assert guarded.throw(SkipIncrement) == 0
assert next(guarded) == 1

try:
    guarded.throw(KeyError, "fatal")
except KeyError:
    pass

assert getgeneratorstate(guarded) == "GEN_CLOSED"

C:\Users\user1\AppData\Local\Temp\ipykernel_16396\3892891329.py:7: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  guarded.throw(KeyError, "fatal")


## 7. `close()` raises `GeneratorExit` inside the generator

Calling `close()` on a suspended generator injects `GeneratorExit`.

The normal pattern is to place cleanup logic in `finally`.

In [36]:
cleanup_log: list[str] = []

def managed_steps() -> Iterator[str]:
    cleanup_log.append("opened")
    try:
        yield "step 1"
        yield "step 2"
    finally:
        cleanup_log.append("closed")

In [37]:
managed = managed_steps()
assert next(managed) == "step 1"
cleanup_log

['opened']

In [38]:
managed.close()
cleanup_log, getgeneratorstate(managed)

(['opened', 'closed'], 'GEN_CLOSED')

### Problem 7 — Record normal completion versus cancellation

Create `tracked_job(steps)`.

- Append each completed step to an event log.
- Append `"job completed"` only after normal exhaustion.
- Append `"job cleaned up"` in every case.
- Closing early must not append `"job completed"`.

### Solution

In [39]:
job_events: list[str] = []

def tracked_job(steps: int) -> Iterator[int]:
    try:
        for step in range(1, steps + 1):
            yield step
            job_events.append(f"completed step {step}")
        job_events.append("job completed")
    finally:
        job_events.append("job cleaned up")

In [40]:
job = tracked_job(3)
assert next(job) == 1
assert next(job) == 2
job.close()
job_events

['completed step 1', 'job cleaned up']

In [41]:
assert job_events == [
    "completed step 1",
    "job cleaned up",
]

## 8. A generator must not yield while handling `GeneratorExit`

A generator that catches `GeneratorExit` and yields another value violates the close protocol.

Python responds with `RuntimeError: generator ignored GeneratorExit`.

In [42]:
def badly_behaved_generator() -> Iterator[str]:
    try:
        yield "running"
    except GeneratorExit:
        yield "this is illegal during close"

bad = badly_behaved_generator()
assert next(bad) == "running"

try:
    bad.close()
except RuntimeError as exc:
    close_protocol_error = str(exc)

close_protocol_error

'generator ignored GeneratorExit'

The generator may still be suspended after the failed close, so we explicitly finish cleaning it up.

In [43]:
try:
    next(bad)
except StopIteration:
    pass

getgeneratorstate(bad)

'GEN_CLOSED'

### Problem 8 — Repair the close protocol

Rewrite the generator so it records that `GeneratorExit` occurred, performs cleanup, and closes without yielding.

### Solution

In [44]:
close_events: list[str] = []

def well_behaved_generator() -> Iterator[str]:
    try:
        yield "running"
    except GeneratorExit:
        close_events.append("GeneratorExit observed")
        raise
    finally:
        close_events.append("cleanup complete")

well_behaved = well_behaved_generator()
assert next(well_behaved) == "running"
well_behaved.close()

assert close_events == ["GeneratorExit observed", "cleanup complete"]
assert getgeneratorstate(well_behaved) == "GEN_CLOSED"

## 9. Delegation with `yield from`

`yield from child` does much more than loop over values.

It forwards:

- `next()`;
- `send()`;
- `throw()`;
- `close()`;
- the child generator's return value.

In [45]:
def child_dialogue() -> Generator[str, str, str]:
    name = yield "What is your name?"
    color = yield f"Hello {name}. Favorite color?"
    return f"{name} likes {color}"


def parent_dialogue() -> Generator[str, str, str]:
    yield "Parent starting"
    summary = yield from child_dialogue()
    yield f"Summary: {summary}"
    return "Parent finished"

In [46]:
dialogue = parent_dialogue()
assert next(dialogue) == "Parent starting"
assert next(dialogue) == "What is your name?"
assert dialogue.send("Ada") == "Hello Ada. Favorite color?"
assert dialogue.send("blue") == "Summary: Ada likes blue"

In [47]:
try:
    next(dialogue)
except StopIteration as exc:
    parent_result = exc.value

parent_result

'Parent finished'

### Problem 9 — Delegate to two subtotal collectors

Create a child generator `subtotal_collector(label)`.

- It starts by yielding `"<label> ready"`.
- Numbers sent to it are accumulated.
- Sending `None` makes it return the subtotal.

Then create `two_stage_total()` that delegates to an `"A"` collector and a `"B"` collector, finally returning their grand total.

### Solution

In [48]:
def subtotal_collector(
    label: str,
) -> Generator[str | float, float | None, float]:
    subtotal = 0.0
    incoming = yield f"{label} ready"

    while incoming is not None:
        subtotal += incoming
        incoming = yield subtotal

    return subtotal


def two_stage_total() -> Generator[str | float, float | None, float]:
    first = yield from subtotal_collector("A")
    second = yield from subtotal_collector("B")
    return first + second

In [49]:
stages = two_stage_total()
assert next(stages) == "A ready"
assert stages.send(10) == 10.0
assert stages.send(2.5) == 12.5
assert stages.send(None) == "B ready"
assert stages.send(4) == 4.0

try:
    stages.send(None)
except StopIteration as exc:
    grand_total = exc.value

grand_total

16.5

In [50]:
assert grand_total == 16.5

## 10. Inspecting the active delegated generator

While a generator is delegating, its `gi_yieldfrom` attribute refers to the active child iterator.

When delegation is not active, `gi_yieldfrom` is `None`.

In [51]:
delegator = parent_dialogue()
assert delegator.gi_yieldfrom is None
assert next(delegator) == "Parent starting"
assert delegator.gi_yieldfrom is None
assert next(delegator) == "What is your name?"

type(delegator.gi_yieldfrom).__name__, getgeneratorstate(delegator.gi_yieldfrom)

('generator', 'GEN_SUSPENDED')

In [52]:
delegator.close()
assert delegator.gi_yieldfrom is None
assert getgeneratorstate(delegator) == "GEN_CLOSED"

### Problem 10 — Report a delegation chain

Write `delegation_chain(generator)`.

It should follow repeated `gi_yieldfrom` references and return the function names of every active generator in the chain.

### Solution

In [53]:
def delegation_chain(generator: Generator[Any, Any, Any]) -> list[str]:
    names: list[str] = []
    current: Any = generator

    while hasattr(current, "gi_code"):
        names.append(current.gi_code.co_name)
        current = current.gi_yieldfrom
        if current is None:
            break

    return names

In [54]:
chain_example = parent_dialogue()
next(chain_example)
next(chain_example)

delegation_chain(chain_example)

['parent_dialogue', 'child_dialogue']

In [55]:
assert delegation_chain(chain_example) == [
    "parent_dialogue",
    "child_dialogue",
]
chain_example.close()

## 11. Inspecting suspended local variables

A suspended generator preserves its frame.

The frame contains local variables, the current instruction position, and the code object.

In [56]:
def weighted_running_total(values: Iterable[int], weight: int) -> Iterator[int]:
    total = 0
    for index, value in enumerate(values):
        weighted = value * weight
        total += weighted
        yield total

In [57]:
weighted = weighted_running_total([3, 5, 7], weight=10)
assert next(weighted) == 30

frame = weighted.gi_frame
assert frame is not None

{
    "state": getgeneratorstate(weighted),
    "function": weighted.gi_code.co_name,
    "locals": dict(frame.f_locals),
    "instruction_offset": frame.f_lasti,
}

{'state': 'GEN_SUSPENDED',
 'function': 'weighted_running_total',
 'locals': {'values': [3, 5, 7],
  'weight': 10,
  'total': 30,
  'index': 0,
  'value': 3,
  'weighted': 30},
 'instruction_offset': 62}

After the generator closes, `gi_frame` becomes `None`.

In [58]:
weighted.close()
weighted.gi_frame, getgeneratorstate(weighted)

(None, 'GEN_CLOSED')

### Problem 11 — Build a reusable generator snapshot

Write `generator_snapshot(generator)` that reports:

- state;
- function name;
- whether it is currently running;
- local variables;
- active delegated iterator type.

### Solution

In [59]:
def generator_snapshot(
    generator: Generator[Any, Any, Any],
) -> dict[str, object]:
    frame = generator.gi_frame
    delegated = generator.gi_yieldfrom

    return {
        "state": getgeneratorstate(generator),
        "function": generator.gi_code.co_name,
        "running": generator.gi_running,
        "locals": {} if frame is None else dict(frame.f_locals),
        "delegating_to": None if delegated is None else type(delegated).__name__,
    }

In [60]:
snapshot_target = weighted_running_total([2, 4], weight=3)
created_snapshot = generator_snapshot(snapshot_target)
assert next(snapshot_target) == 6
suspended_snapshot = generator_snapshot(snapshot_target)
snapshot_target.close()
closed_snapshot = generator_snapshot(snapshot_target)

created_snapshot, suspended_snapshot, closed_snapshot

({'state': 'GEN_CREATED',
  'function': 'weighted_running_total',
  'running': False,
  'locals': {'values': [2, 4], 'weight': 3},
  'delegating_to': None},
 {'state': 'GEN_SUSPENDED',
  'function': 'weighted_running_total',
  'running': False,
  'locals': {'values': [2, 4],
   'weight': 3,
   'total': 6,
   'index': 0,
   'value': 2,
   'weighted': 6},
  'delegating_to': None},
 {'state': 'GEN_CLOSED',
  'function': 'weighted_running_total',
  'running': False,
  'locals': {},
  'delegating_to': None})

In [61]:
assert created_snapshot["state"] == "GEN_CREATED"
assert suspended_snapshot["state"] == "GEN_SUSPENDED"
assert suspended_snapshot["locals"]["total"] == 6
assert closed_snapshot["state"] == "GEN_CLOSED"
assert closed_snapshot["locals"] == {}

## 12. Observing `GEN_RUNNING` without a global generator variable

The original style of demonstration often stores the generator in a global variable.

A cleaner experiment can pass a callback into the generator. The callback looks up the generator through a small holder object only while the generator body is running.

In [62]:
running_holder: dict[str, Generator[str, None, None]] = {}
running_observations: list[str] = []

def observe_running() -> None:
    running_observations.append(
        getgeneratorstate(running_holder["generator"])
    )


def callback_observed_generator() -> Iterator[str]:
    observe_running()
    yield "paused"

In [63]:
running_holder["generator"] = callback_observed_generator()
assert next(running_holder["generator"]) == "paused"
running_observations

['GEN_RUNNING']

### Problem 12 — Record every visible lifecycle state

Create a generator that records `GEN_RUNNING` internally and let the caller record the other states externally.

The final ordered list should be:

`GEN_CREATED`, `GEN_RUNNING`, `GEN_SUSPENDED`, `GEN_CLOSED`.

### Solution

In [64]:
visible_states: list[str] = []
state_holder: dict[str, Generator[int, None, None]] = {}

def all_state_demo() -> Iterator[int]:
    visible_states.append(getgeneratorstate(state_holder["generator"]))
    yield 1

state_holder["generator"] = all_state_demo()
visible_states.append(getgeneratorstate(state_holder["generator"]))
next(state_holder["generator"])
visible_states.append(getgeneratorstate(state_holder["generator"]))
state_holder["generator"].close()
visible_states.append(getgeneratorstate(state_holder["generator"]))

visible_states

['GEN_CREATED', 'GEN_RUNNING', 'GEN_SUSPENDED', 'GEN_CLOSED']

In [65]:
assert visible_states == [
    "GEN_CREATED",
    "GEN_RUNNING",
    "GEN_SUSPENDED",
    "GEN_CLOSED",
]

## 13. Generator re-entrancy

A running generator cannot be resumed again before it reaches another suspension point.

Attempting to do so raises `ValueError: generator already executing`.

In [66]:
reentrant_holder: dict[str, Generator[str, None, None]] = {}

def reentrant_example() -> Iterator[str]:
    try:
        next(reentrant_holder["generator"])
    except ValueError as exc:
        yield str(exc)

reentrant_holder["generator"] = reentrant_example()
next(reentrant_holder["generator"])

'generator already executing'

### Problem 13 — Avoid re-entrancy with a command queue

Instead of calling a generator from inside its own callback, queue follow-up commands and process them after the generator suspends.

Create `queued_processor(initial_commands)` that yields processed commands. A command ending with `"!"` should queue a lowercase follow-up command.

### Solution

In [67]:
def queued_processor(initial_commands: Iterable[str]) -> Iterator[str]:
    pending = deque(initial_commands)

    while pending:
        command = pending.popleft()
        yield command

        if command.endswith("!"):
            pending.append(command[:-1].lower())

In [68]:
assert list(queued_processor(["A!", "B", "C!"])) == [
    "A!",
    "B",
    "C!",
    "a",
    "c",
]

## 14. Laziness is pull-based backpressure

A normal generator produces a value only when the consumer requests one.

This is a simple form of backpressure: the consumer controls the production rate.

In [69]:
production_log: list[int] = []

def instrumented_source(limit: int) -> Iterator[int]:
    for number in range(limit):
        production_log.append(number)
        yield number

In [70]:
source = instrumented_source(1_000_000)
first_three = list(islice(source, 3))
first_three, production_log

([0, 1, 2], [0, 1, 2])

Only three source values were produced, even though the conceptual stream contains one million values.

### Problem 14 — Add one-item look-ahead

Write `with_last_flag(iterable)`.

It should yield `(item, is_last)` while reading at most one item ahead.

### Solution

In [71]:
_sentinel = object()

def with_last_flag(iterable: Iterable[T]) -> Iterator[tuple[T, bool]]:
    iterator = iter(iterable)
    current = next(iterator, _sentinel)

    if current is _sentinel:
        return

    while True:
        following = next(iterator, _sentinel)
        is_last = following is _sentinel
        yield current, is_last

        if is_last:
            return

        current = following

In [72]:
assert list(with_last_flag([10, 20, 30])) == [
    (10, False),
    (20, False),
    (30, True),
]
assert list(with_last_flag([])) == []

## 15. Updating a suspended generator configuration

A generator may use values sent after each yield to update future behavior.

This creates a stateful stream transducer.

In [73]:
def adaptive_threshold(
    values: Iterable[int],
    initial_threshold: int,
) -> Generator[int, int | None, int]:
    threshold = initial_threshold
    accepted = 0

    for value in values:
        if value >= threshold:
            accepted += 1
            new_threshold = yield value
            if new_threshold is not None:
                threshold = new_threshold

    return accepted

In [74]:
adaptive = adaptive_threshold([3, 10, 7, 20, 15], initial_threshold=8)
assert next(adaptive) == 10
assert adaptive.send(18) == 20

The value `7` was rejected under the old threshold. After the update, `15` is also rejected.

In [75]:
try:
    next(adaptive)
except StopIteration as exc:
    accepted_count = exc.value

accepted_count

2

### Problem 15 — Build an adjustable stride generator

Create `adjustable_stride(values, initial_stride)`.

- Yield every `stride`-th value.
- After each yield, the caller may send a new positive stride.
- Return the number of values yielded.

### Solution

In [76]:
def adjustable_stride(
    values: Iterable[T],
    initial_stride: int,
) -> Generator[T, int | None, int]:
    if initial_stride <= 0:
        raise ValueError("stride must be positive")

    stride = initial_stride
    yielded = 0

    for index, value in enumerate(values):
        if index % stride == 0:
            yielded += 1
            new_stride = yield value

            if new_stride is not None:
                if new_stride <= 0:
                    raise ValueError("stride must be positive")
                stride = new_stride

    return yielded

In [77]:
stride_generator = adjustable_stride(range(10), initial_stride=3)
assert next(stride_generator) == 0
assert stride_generator.send(None) == 3
assert stride_generator.send(2) == 4
assert stride_generator.send(None) == 6
assert stride_generator.send(None) == 8

try:
    next(stride_generator)
except StopIteration as exc:
    stride_yield_count = exc.value

stride_yield_count

5

In [78]:
assert stride_yield_count == 5

## 16. Generator expressions also have states

A generator expression creates a real generator object, so `getgeneratorstate()` works on it.

In [79]:
squares = (number * number for number in range(3))
getgeneratorstate(squares)

'GEN_CREATED'

In [80]:
next(squares), getgeneratorstate(squares)

(0, 'GEN_SUSPENDED')

In [81]:
list(squares), getgeneratorstate(squares)

([1, 4], 'GEN_CLOSED')

Generator expressions evaluate lazily. External variables may therefore change before iteration begins.

In [82]:
multiplier_value = 2
lazy_products = (number * multiplier_value for number in range(3))
multiplier_value = 10
list(lazy_products)

[0, 10, 20]

### Problem 16 — Freeze configuration intentionally

Write `make_scaled_values(values, scale)` so later changes to an external variable cannot alter the chosen scale.

### Solution

In [83]:
def make_scaled_values(values: Iterable[int], scale: int) -> Iterator[int]:
    frozen_scale = scale
    return (value * frozen_scale for value in values)

external_scale = 3
scaled = make_scaled_values(range(4), external_scale)
external_scale = 100

assert list(scaled) == [0, 3, 6, 9]

## 17. `StopIteration` inside a generator body

A generator should terminate with `return`, not by manually raising `StopIteration`.

An escaping `StopIteration` from inside a generator is transformed into `RuntimeError`.

In [84]:
def incorrect_termination() -> Iterator[int]:
    yield 1
    raise StopIteration("do not do this")

broken = incorrect_termination()
assert next(broken) == 1

try:
    next(broken)
except RuntimeError as exc:
    transformed_error = str(exc)

transformed_error

'generator raised StopIteration'

### Problem 17 — Safely stop at the first missing item

Write `until_missing(iterable, missing)`.

Yield values until `missing` appears, then terminate with `return`. Do not raise `StopIteration` manually.

### Solution

In [85]:
def until_missing(iterable: Iterable[T], missing: T) -> Iterator[T]:
    for item in iterable:
        if item == missing:
            return
        yield item

assert list(until_missing([1, 2, None, 3], None)) == [1, 2]

## 18. Generator-based context managers

`contextlib.contextmanager` converts a one-yield generator into a context manager.

- Code before `yield` behaves like `__enter__`.
- The yielded value is bound after `as`.
- Code in `finally` behaves like `__exit__` cleanup.

In [86]:
context_events: list[str] = []

@contextmanager
def temporary_label(label: str) -> Iterator[str]:
    context_events.append(f"enter:{label}")
    try:
        yield label.upper()
    finally:
        context_events.append(f"exit:{label}")

In [87]:
with temporary_label("demo") as active_label:
    context_events.append(f"inside:{active_label}")

context_events

['enter:demo', 'inside:DEMO', 'exit:demo']

### Problem 18 — Create a reversible list mutation

Build a generator-based context manager `temporary_append(target, value)`.

It should append a value on entry and remove that exact appended element on exit, even if the block raises an exception.

### Solution

In [88]:
@contextmanager
def temporary_append(target: list[T], value: T) -> Iterator[list[T]]:
    target.append(value)
    try:
        yield target
    finally:
        removed = target.pop()
        if removed != value:
            raise RuntimeError("target list was structurally modified")

In [89]:
items = [1, 2]

try:
    with temporary_append(items, 99) as active_items:
        assert active_items == [1, 2, 99]
        raise ValueError("simulated block failure")
except ValueError:
    pass

items

[1, 2]

In [90]:
assert items == [1, 2]

## 19. Recursive delegation with return values

A recursive generator can both yield traversal values and return aggregate information to its parent through `yield from`.

In [91]:
tree = {
    "root": ["left", "right"],
    "left": ["left.leaf"],
    "right": ["right.a", "right.b"],
    "left.leaf": [],
    "right.a": [],
    "right.b": [],
}

### Problem 19 — Yield preorder nodes and return the leaf count

Create `walk_and_count_leaves(tree, node)`.

- Yield each node in preorder.
- Return the number of leaves in that node's subtree.

### Solution

In [92]:
def walk_and_count_leaves(
    tree: dict[str, list[str]],
    node: str,
) -> Generator[str, None, int]:
    yield node
    children = tree.get(node, [])

    if not children:
        return 1

    leaf_count = 0
    for child in children:
        leaf_count += yield from walk_and_count_leaves(tree, child)

    return leaf_count

In [93]:
visited_nodes, leaf_count = consume_with_result(
    walk_and_count_leaves(tree, "root")
)

visited_nodes, leaf_count

(['root', 'left', 'left.leaf', 'right', 'right.a', 'right.b'], 3)

In [94]:
assert visited_nodes == [
    "root",
    "left",
    "left.leaf",
    "right",
    "right.a",
    "right.b",
]
assert leaf_count == 3

## 20. Cooperative tasks and generator states

A cooperative task can perform a small amount of work, yield control, and resume later.

A scheduler can observe each task's generator state between steps.

In [95]:
def task(name: str, steps: int) -> Generator[str, None, str]:
    for step in range(1, steps + 1):
        yield f"{name}:{step}"
    return f"{name}:done"

### Problem 20 — Build a state-aware cooperative scheduler

Write `run_tasks(*tasks)`.

It should:

- resume each active task once per round;
- record the state before and after every resume;
- collect every yielded event;
- preserve each task's final return value.

### Solution

In [96]:
def run_tasks(
    *tasks: Generator[str, None, str],
) -> dict[str, object]:
    active = deque(enumerate(tasks))
    timeline: list[dict[str, object]] = []
    results: dict[int, str] = {}

    while active:
        task_id, generator = active.popleft()
        before = getgeneratorstate(generator)

        try:
            event = next(generator)
        except StopIteration as exc:
            results[task_id] = exc.value
            timeline.append(
                {
                    "task": task_id,
                    "before": before,
                    "event": None,
                    "after": getgeneratorstate(generator),
                }
            )
        else:
            timeline.append(
                {
                    "task": task_id,
                    "before": before,
                    "event": event,
                    "after": getgeneratorstate(generator),
                }
            )
            active.append((task_id, generator))

    return {"timeline": timeline, "results": results}

In [97]:
schedule = run_tasks(task("A", 2), task("B", 3))
schedule

{'timeline': [{'task': 0,
   'before': 'GEN_CREATED',
   'event': 'A:1',
   'after': 'GEN_SUSPENDED'},
  {'task': 1,
   'before': 'GEN_CREATED',
   'event': 'B:1',
   'after': 'GEN_SUSPENDED'},
  {'task': 0,
   'before': 'GEN_SUSPENDED',
   'event': 'A:2',
   'after': 'GEN_SUSPENDED'},
  {'task': 1,
   'before': 'GEN_SUSPENDED',
   'event': 'B:2',
   'after': 'GEN_SUSPENDED'},
  {'task': 0, 'before': 'GEN_SUSPENDED', 'event': None, 'after': 'GEN_CLOSED'},
  {'task': 1,
   'before': 'GEN_SUSPENDED',
   'event': 'B:3',
   'after': 'GEN_SUSPENDED'},
  {'task': 1,
   'before': 'GEN_SUSPENDED',
   'event': None,
   'after': 'GEN_CLOSED'}],
 'results': {0: 'A:done', 1: 'B:done'}}

In [98]:
assert [row["event"] for row in schedule["timeline"] if row["event"]] == [
    "A:1",
    "B:1",
    "A:2",
    "B:2",
    "B:3",
]
assert schedule["results"] == {0: "A:done", 1: "B:done"}

## 21. One-shot generators and shared-state bugs

A generator object is an iterator and is therefore one-shot.

Storing one generator object where a reusable iterable is expected can create subtle shared-state bugs.

In [99]:
shared = (number for number in range(4))
first_consumer = list(islice(shared, 2))
second_consumer = list(shared)

first_consumer, second_consumer

([0, 1], [2, 3])

### Problem 21 — Replace a shared generator with a factory

Create a reusable object that can produce a fresh square generator for each consumer.

### Solution

In [100]:
class SquareStream:
    def __init__(self, limit: int) -> None:
        self.limit = limit

    def __iter__(self) -> Iterator[int]:
        return (number * number for number in range(self.limit))

square_stream = SquareStream(4)
assert list(square_stream) == [0, 1, 4, 9]
assert list(square_stream) == [0, 1, 4, 9]

## 22. Testing generator state transitions

A generator test should verify more than final output.

For advanced generators, also test:

- the initial state;
- suspension after each protocol step;
- cleanup after `close()`;
- the final return value;
- exceptional paths.

### Problem 22 — Write a state assertion helper

Create `assert_generator_state(generator, expected)` that raises a useful `AssertionError` when the state differs.

### Solution

In [101]:
def assert_generator_state(
    generator: Generator[Any, Any, Any],
    expected: str,
) -> None:
    actual = getgeneratorstate(generator)
    if actual != expected:
        raise AssertionError(
            f"expected generator state {expected}, received {actual}"
        )

In [102]:
test_generator = multiplier_protocol(3)
assert_generator_state(test_generator, "GEN_CREATED")
assert next(test_generator) == "READY"
assert_generator_state(test_generator, "GEN_SUSPENDED")
test_generator.close()
assert_generator_state(test_generator, "GEN_CLOSED")

## 23. Finite-state machines implemented as generators

A suspended generator naturally stores state between inputs.

This makes it useful for small protocol parsers and finite-state machines.

### Problem 23 — Track balanced parentheses

Create `parenthesis_tracker()`.

Protocol:

- Prime it to receive the initial depth `0`.
- Send one character at a time.
- It yields the current nesting depth.
- Sending the sentinel `END` terminates it.
- It returns the maximum depth.
- It raises `ValueError` for an unexpected closing parenthesis or unfinished input.

### Solution

In [103]:
END = object()

def parenthesis_tracker() -> Generator[int, str | object, int]:
    depth = 0
    maximum_depth = 0
    token = yield depth

    while token is not END:
        if token == "(":
            depth += 1
            maximum_depth = max(maximum_depth, depth)
        elif token == ")":
            depth -= 1
            if depth < 0:
                raise ValueError("unexpected closing parenthesis")

        token = yield depth

    if depth != 0:
        raise ValueError("unclosed parenthesis")

    return maximum_depth

In [104]:
tracker = parenthesis_tracker()
assert next(tracker) == 0
assert tracker.send("(") == 1
assert tracker.send("a") == 1
assert tracker.send("(") == 2
assert tracker.send(")") == 1
assert tracker.send(")") == 0

try:
    tracker.send(END)
except StopIteration as exc:
    maximum_depth = exc.value

maximum_depth

2

In [105]:
assert maximum_depth == 2
assert getgeneratorstate(tracker) == "GEN_CLOSED"

## 24. Capstone — A command-driven streaming monitor

The capstone combines state, `send()`, `throw()`, return values, validation, and dynamic configuration.

The monitor maintains a bounded window of numeric samples.

### Capstone problem

Implement `streaming_monitor(window_size)`.

The first yielded value is an empty statistics dictionary.

Commands sent to the generator:

- a number: add it and yield updated statistics;
- `("resize", n)`: change the window size and preserve the newest samples;
- `"reset"`: clear the window;
- `"stop"`: return the total number of numeric samples processed;
- an injected `IgnoreSample` exception: leave state unchanged and yield current statistics.

Statistics must contain `count`, `minimum`, `maximum`, and `average`.

### Solution

In [106]:
class IgnoreSample(Exception):
    pass


def _statistics(samples: deque[float]) -> dict[str, float | int | None]:
    if not samples:
        return {
            "count": 0,
            "minimum": None,
            "maximum": None,
            "average": None,
        }

    return {
        "count": len(samples),
        "minimum": min(samples),
        "maximum": max(samples),
        "average": sum(samples) / len(samples),
    }


def streaming_monitor(
    window_size: int,
) -> Generator[dict[str, float | int | None], object, int]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    samples: deque[float] = deque(maxlen=window_size)
    processed = 0
    current = _statistics(samples)

    while True:
        try:
            command = yield current
        except IgnoreSample:
            current = _statistics(samples)
            continue

        if command == "stop":
            return processed

        if command == "reset":
            samples.clear()

        elif (
            isinstance(command, tuple)
            and len(command) == 2
            and command[0] == "resize"
        ):
            new_size = command[1]
            if not isinstance(new_size, int) or new_size <= 0:
                raise ValueError("new window size must be a positive integer")
            samples = deque(list(samples)[-new_size:], maxlen=new_size)

        elif isinstance(command, (int, float)) and not isinstance(command, bool):
            samples.append(float(command))
            processed += 1

        else:
            raise TypeError(f"unsupported command: {command!r}")

        current = _statistics(samples)

We create and prime the monitor.

In [107]:
monitor = streaming_monitor(window_size=3)
initial_stats = next(monitor)
initial_stats

{'count': 0, 'minimum': None, 'maximum': None, 'average': None}

We send samples. Once the window is full, the oldest sample is discarded automatically.

In [108]:
stats_10 = monitor.send(10)
stats_20 = monitor.send(20)
stats_30 = monitor.send(30)
stats_40 = monitor.send(40)

stats_10, stats_20, stats_30, stats_40

({'count': 1, 'minimum': 10.0, 'maximum': 10.0, 'average': 10.0},
 {'count': 2, 'minimum': 10.0, 'maximum': 20.0, 'average': 15.0},
 {'count': 3, 'minimum': 10.0, 'maximum': 30.0, 'average': 20.0},
 {'count': 3, 'minimum': 20.0, 'maximum': 40.0, 'average': 30.0})

We resize the window while the generator is suspended.

In [109]:
resized_stats = monitor.send(("resize", 2))
resized_stats

{'count': 2, 'minimum': 30.0, 'maximum': 40.0, 'average': 35.0}

An injected recoverable exception leaves the current window unchanged.

In [110]:
ignored_stats = monitor.throw(IgnoreSample)
ignored_stats, getgeneratorstate(monitor)

({'count': 2, 'minimum': 30.0, 'maximum': 40.0, 'average': 35.0},
 'GEN_SUSPENDED')

A reset clears the rolling statistics without resetting the lifetime processed counter.

In [111]:
reset_stats = monitor.send("reset")
reset_stats

{'count': 0, 'minimum': None, 'maximum': None, 'average': None}

Finally, the stop command closes the generator and returns the lifetime sample count.

In [112]:
try:
    monitor.send("stop")
except StopIteration as exc:
    lifetime_processed = exc.value

lifetime_processed, getgeneratorstate(monitor)

(4, 'GEN_CLOSED')

In [113]:
assert initial_stats == {
    "count": 0,
    "minimum": None,
    "maximum": None,
    "average": None,
}
assert stats_30 == {
    "count": 3,
    "minimum": 10.0,
    "maximum": 30.0,
    "average": 20.0,
}
assert stats_40 == {
    "count": 3,
    "minimum": 20.0,
    "maximum": 40.0,
    "average": 30.0,
}
assert resized_stats == {
    "count": 2,
    "minimum": 30.0,
    "maximum": 40.0,
    "average": 35.0,
}
assert ignored_stats == resized_stats
assert reset_stats["count"] == 0
assert lifetime_processed == 4
assert getgeneratorstate(monitor) == "GEN_CLOSED"

## Final review

At this point, you have used generator states as part of real control-flow design.

You should now be able to explain:

1. why calling a generator function does not execute its body;
2. how a value sent by `send()` becomes the result of `yield`;
3. why a newly created generator accepts only `None`;
4. how a generator return value travels through `StopIteration.value`;
5. how `throw()` differs from raising an exception outside the generator;
6. why `close()` should be paired with `try/finally`;
7. why yielding during `GeneratorExit` is illegal;
8. what `yield from` forwards automatically;
9. how `gi_frame`, `gi_code`, `gi_running`, and `gi_yieldfrom` help debugging;
10. why a running generator cannot be re-entered;
11. how lazy iteration implements consumer-controlled backpressure;
12. how generators can implement protocols, context managers, state machines, and cooperative tasks.

## Suggested experiments

Try modifying the notebook in these ways:

- Throw an unhandled exception into the capstone monitor and inspect its final state.
- Close `two_stage_total()` while it is delegating and inspect both parent and child states.
- Add a `pause` command to the cooperative scheduler.
- Extend the parenthesis tracker to handle square and curly brackets.
- Add median calculation to the streaming monitor.
- Write a test that verifies cleanup after every possible suspension point.